AI-Based Multi-Factor Agricultural Crop Yield Prediction Using Climate and Soil Data Analytics

Load the dataset

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/shrih/OneDrive/Desktop/project/agriculture_ml_dataset_50000_rows.csv")

df.columns

Index(['temperature_c', 'rainfall_mm', 'humidity_percent',
       'solar_radiation_w_m2', 'soil_nutrients_index', 'soil_moisture_percent',
       'soil_ph', 'organic_matter_percent', 'electrical_conductivity_ds_m',
       'soil_type', 'area_type', 'crop_yield_ton_per_hectare'],
      dtype='object')

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   temperature_c                 50000 non-null  float64
 1   rainfall_mm                   50000 non-null  float64
 2   humidity_percent              50000 non-null  float64
 3   solar_radiation_w_m2          50000 non-null  float64
 4   soil_nutrients_index          50000 non-null  float64
 5   soil_moisture_percent         50000 non-null  float64
 6   soil_ph                       50000 non-null  float64
 7   organic_matter_percent        50000 non-null  float64
 8   electrical_conductivity_ds_m  50000 non-null  float64
 9   soil_type                     50000 non-null  object 
 10  area_type                     50000 non-null  object 
 11  crop_yield_ton_per_hectare    50000 non-null  float64
dtypes: float64(10), object(2)
memory usage: 4.6+ MB


In [23]:
df.describe(include="all")

,temperature_c,rainfall_mm,humidity_percent,solar_radiation_w_m2,soil_nutrients_index,soil_moisture_percent,soil_ph,organic_matter_percent,electrical_conductivity_ds_m,soil_type,area_type,crop_yield_ton_per_hectare
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000,50000,50000.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,6,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,clay,tropical,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8334,8390,NaN
mean,27.505200,1358.367624,66.431857,236.918258,60.096323,34.996865,6.501293,3.501799,1.203095,NaN,NaN,1.960058
std,5.501904,708.050898,18.258876,48.203985,19.870876,11.906627,0.881298,1.203818,0.580944,NaN,NaN,0.596896
min,5.270000,0.000000,10.000000,100.000000,10.000000,5.000000,4.500000,0.500000,0.100000,NaN,NaN,0.500000
25%,24.230000,839.415000,56.650000,202.210000,46.537500,26.900000,5.890000,2.680000,0.790000,NaN,NaN,1.550000
50%,27.740000,1445.320000,70.950000,232.380000,60.090000,34.960000,6.500000,3.500000,1.200000,NaN,NaN,1.960000
75%,31.020000,1881.870000,79.600000,268.590000,73.692500,43.040000,7.110000,4.320000,1.600000,NaN,NaN,2.370000


Data Cleaning

In [24]:
df.isnull().sum()

temperature_c                   0
rainfall_mm                     0
humidity_percent                0
solar_radiation_w_m2            0
soil_nutrients_index            0
soil_moisture_percent           0
soil_ph                         0
organic_matter_percent          0
electrical_conductivity_ds_m    0
soil_type                       0
area_type                       0
crop_yield_ton_per_hectare      0
dtype: int64

Encode Categorical Features(soiltype,area type)

One-Hot Encoding

overall climate index

In [25]:
df["climate_suitability_index"] = (
    df["temperature_c"] +
    df["rainfall_mm"] +
    df["humidity_percent"]
) / 3

Soil Fertility Index

In [26]:
df["soil_fertility_index"] = (
    df["soil_nutrients_index"] * 0.5 +
    df["organic_matter_percent"] * 10 -
    df["electrical_conductivity_ds_m"] * 2
)

Water Availability Index

In [27]:
df["water_availability_score"] = (
    df["rainfall_mm"] +
    df["soil_moisture_percent"]
) / 2

Crop Stress Index

In [28]:
df["crop_stress_index"] = (
    df["temperature_c"] * 0.3 +
    df["solar_radiation_w_m2"] * 0.002 -
    df["humidity_percent"] * 0.1
)

NDVI Index (Normalized Difference Vegetation Index)

In [29]:
df["ndvi_index"] = (
    0.002 * df["solar_radiation_w_m2"]
    + 0.003 * df["soil_moisture_percent"]
    + 0.002 * df["humidity_percent"]
) / 2

In [30]:
df["fertilizer_usage_kg_per_hectare"] = np.random.normal(150, 40, len(df))
df["planting_density_plants_per_hectare"] = np.random.randint(20000, 100000, len(df))

In [31]:
df["irrigation_type"] = np.random.choice(
    ["drip","sprinkler","flood","rainfed"], len(df)
)

df["crop_type"] = np.random.choice(
    ["rice","wheat","maize","cotton","millet"], len(df)
)

soil-Health score

In [32]:
df["soil_health_score"] = (
    df["soil_nutrients_index"] +
    df["organic_matter_percent"] +
    df["soil_ph"]
) / 3

growth condition score(climate,soil,water)

In [33]:
df["growth_condition_index"] = (
    df["climate_suitability_index"] +
    df["soil_health_score"] +
    df["water_availability_score"]
) / 3

Yield Efficiency Index(crop yield,fertilizer usage)

In [34]:
df["yield_efficiency_index"] = (
    df["crop_yield_ton_per_hectare"] /
    df["fertilizer_usage_kg_per_hectare"]
)

Feature Engineering

In [35]:
df_encoded = pd.get_dummies(
    df,
    columns=["soil_type", "area_type", "crop_type", "irrigation_type"]
)

Feature Scaling

In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = df.select_dtypes(include=['float64','int64']).columns

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

cleaned dataset

In [37]:
df.to_csv("agriculture_dataset_feature_engineered.csv", index=False)

In [38]:
df.columns


Index(['temperature_c', 'rainfall_mm', 'humidity_percent',
       'solar_radiation_w_m2', 'soil_nutrients_index', 'soil_moisture_percent',
       'soil_ph', 'organic_matter_percent', 'electrical_conductivity_ds_m',
       'soil_type', 'area_type', 'crop_yield_ton_per_hectare',
       'climate_suitability_index', 'soil_fertility_index',
       'water_availability_score', 'crop_stress_index', 'ndvi_index',
       'fertilizer_usage_kg_per_hectare',
       'planting_density_plants_per_hectare', 'irrigation_type', 'crop_type',
       'soil_health_score', 'growth_condition_index',
       'yield_efficiency_index'],
      dtype='object')

In [39]:
df.shape

(50000, 24)

In [40]:
# =========================
# Import Libraries
# =========================
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# =========================
# Load Dataset
# =========================
df = pd.read_csv("C:/Users/shrih/OneDrive/Desktop/project/agriculture_dataset_feature_engineered.csv")

# =========================
# Data Cleaning
# =========================
df = df.dropna()
df = df.drop_duplicates()

# =========================
# Define Features and Target
# =========================
y = df["crop_yield_ton_per_hectare"]
X = df.drop("crop_yield_ton_per_hectare", axis=1)

# =========================
# Encode Categorical Data
# =========================
X = pd.get_dummies(X)

# =========================
# Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# XGBoost Model
# =========================
model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# =========================
# Train Model
# =========================
model.fit(X_train, y_train)

# =========================
# Prediction
# =========================
y_pred = model.predict(X_test)

# =========================
# Evaluation
# =========================
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("R2 Score (Accuracy):", r2)
print("Mean Absolute Error:", mae)

# =========================
# Save Model
# =========================
import joblib

joblib.dump(model, "yield_model.pkl")
joblib.dump(X.columns, "model_columns.pkl")

R2 Score (Accuracy): 0.9968266652829318
Mean Absolute Error: 0.027438440905300503


['model_columns.pkl']